# Loading, Extending, and Merging Public Ontologies with owlready2

In biomedical informatics, you rarely build ontologies from scratch. Instead, you load existing public ontologies — such as the Gene Ontology, Disease Ontology, or Human Phenotype Ontology — and extend or combine them for your specific use case.

This notebook walks through three core skills:
1. **Loading** publicly available OWL ontologies (from URLs and local files)
2. **Extending** an existing ontology with new classes, properties, and restrictions
3. **Merging** information from two separate ontologies via imports and cross-references

We use two real biomedical ontologies:
- **GO slim** (Gene Ontology generic slim) — 140 classes covering biological processes, molecular functions, and cellular components
- **Disease Ontology subset** — 77 classes covering cancer types (breast, hematologic, lung, brain)

**Prerequisites:** Familiarity with OWL basics (classes, properties, restrictions) and the owlready2 library.

## Section 1: Setup & Loading from a URL

Public ontologies are typically hosted as OWL/RDF files at stable URLs. owlready2 can load directly from a URL using `get_ontology(url).load()`.

Let's demonstrate loading the Gene Ontology slim directly from the GO Consortium's servers.

In [25]:
from owlready2 import *
import os

# Ensure Java is available for the reasoner (Section 8)
# This points to the JVM installed in the BMDSOWL environment
java_home = os.path.join(os.environ.get("CONDA_PREFIX", ""), "lib", "jvm")
if os.path.exists(java_home):
    os.environ["JAVA_HOME"] = java_home
    os.environ["PATH"] = os.path.join(java_home, "bin") + ":" + os.environ["PATH"]

The GO Consortium hosts ontology subsets ("slims") at predictable URLs. Here we load one directly over the network:

In [26]:
# Loading an ontology from a public URL
# NOTE: This requires network access and may take a few seconds
go_url = "https://current.geneontology.org/ontology/subsets/goslim_generic.owl"

try:
    go_remote = get_ontology(go_url).load()
    print(f"Loaded from URL successfully!")
    print(f"  IRI: {go_remote.base_iri}")
    print(f"  Classes: {len(list(go_remote.classes()))}")
except Exception as e:
    print(f"URL loading failed (network issue): {e}")
    print("We'll use the local copy instead — see next section.")

URL loading failed (network issue): Cannot download 'https://current.geneontology.org/ontology/subsets/goslim_generic.owl'!
We'll use the local copy instead — see next section.


**What happened:**
- `get_ontology(url)` creates an ontology object bound to that IRI
- `.load()` fetches the file (HTTP GET) and parses it into owlready2's internal graph
- If the network is unavailable, we handle the error gracefully

In practice, you'll often download ontology files once and load them locally for speed and reproducibility. That's what we'll do for the rest of this tutorial.

## Section 2: Loading from a Local File

For reproducibility and speed, it's best to bundle ontology files locally. The `data/` folder contains two pre-downloaded ontologies:
- `goslim_generic.owl` — the same GO slim we loaded above
- `disease_ontology_subset.owl` — a curated subset of the Human Disease Ontology (DOID) focused on cancer types

In [27]:
# Loading from a local file — use the file:// protocol or a relative path
go = get_ontology("file://data/goslim_generic.owl").load()

print(f"GO Slim loaded from local file")
print(f"  Base IRI: {go.base_iri}")
print(f"  Number of classes: {len(list(go.classes()))}")
print(f"  Number of object properties: {len(list(go.object_properties()))}")
print(f"  Number of annotation properties: {len(list(go.annotation_properties()))}")

GO Slim loaded from local file
  Base IRI: http://purl.obolibrary.org/obo/go/subsets/goslim_generic.owl#
  Number of classes: 150
  Number of object properties: 10
  Number of annotation properties: 67


**Key points:**
- `file://` prefix tells owlready2 to read from the filesystem
- The IRI (Internationalized Resource Identifier) is embedded in the OWL file itself — it identifies the ontology regardless of where the file lives
- owlready2 supports `.owl` (RDF/XML), `.nt` (N-Triples), and `.ntriples` formats

## Section 3: Exploring an Existing Ontology

Before extending or merging an ontology, you need to understand its structure. Let's explore the GO slim.

In [28]:
# List all object properties (relationships between classes)
print("Object Properties in GO slim:")
print("-" * 40)
for prop in go.object_properties():
    label = prop.label[0] if prop.label else prop.name
    print(f"  {label} ({prop.name})")

Object Properties in GO slim:
----------------------------------------
  part of (BFO_0000050)
  has part (BFO_0000051)
  occurs in (BFO_0000066)
  starts during (RO_0002091)
  happens during (RO_0002092)
  ends during (RO_0002093)
  regulates (RO_0002211)
  negatively regulates (RO_0002212)
  positively regulates (RO_0002213)
  occurs in tissue (occurs_in_tissue)


In [29]:
# Explore the class hierarchy — find top-level classes (direct children of Thing)
top_classes = [c for c in go.classes() if Thing in c.is_a]
print(f"Top-level classes (direct subclasses of owl:Thing): {len(top_classes)}")
print()

# Show a sample with their labels
for cls in top_classes[:15]:
    label = cls.label[0] if cls.label else cls.name
    print(f"  {label}")
print(f"  ... and {len(top_classes) - 15} more")

Top-level classes (direct subclasses of owl:Thing): 106

  mitotic cell cycle
  mitotic nuclear division
  cytokinesis
  membrane organization
  virus receptor activity
  cytoplasmic translation
  protein-containing complex assembly
  immune system process
  muscle system process
  circulatory system process
  renal system process
  respiratory system process
  DNA binding
  RNA binding
  cytoskeletal motor activity
  ... and 91 more


In [30]:
# Search by label — find all classes related to "transport"
transport_classes = go.search(label="*transport*")
print("Classes matching 'transport':")
for cls in transport_classes:
    if hasattr(cls, 'label'):
        print(f"  {cls.label[0] if cls.label else cls.name}")

Classes matching 'transport':
  SERINC3,5,(1,2,4) transport L-Ser from cytosol to plasma membrane
  MMADHC targets transport of cytosolic cob(II)alamin to mitochondria
  Miscellaneous transport and binding events
  SLC-mediated transmembrane transport
  ABC-family proteins mediated transport
  transporter activity
  intracellular protein transport
  nucleocytoplasmic transport
  transmembrane transport
  vesicle-mediated transport


In [31]:
# Inspect a specific class in detail
catalytic = go.search_one(label="catalytic activity")
if catalytic:
    print(f"Class: {catalytic.name}")
    print(f"  Label: {catalytic.label}")
    print(f"  IRI: {catalytic.iri}")
    print(f"  Parents: {catalytic.is_a}")
    print(f"  Subclasses: {list(catalytic.subclasses())}")
    
    # Check for annotations
    if catalytic.comment:
        print(f"  Comment: {catalytic.comment}")

Class: GO_0003824
  Label: ['catalytic activity']
  IRI: http://purl.obolibrary.org/obo/GO_0003824
  Parents: [owl.Thing]
  Subclasses: [obo.GO_0016740, obo.GO_0140096, obo.GO_0140098, obo.GO_0016787, obo.GO_0009975, obo.GO_0016491, obo.GO_0016829, obo.GO_0016853, obo.GO_0016874, obo.GO_0140097]


**Exploration patterns:**
- `onto.classes()` — iterate all classes
- `onto.object_properties()` — iterate all object properties
- `onto.search(label="*keyword*")` — wildcard search on labels
- `onto.search_one(label="exact match")` — find a single class
- `cls.is_a` — parent classes and restrictions
- `cls.subclasses()` — direct children in the hierarchy

## Section 4: Extending with New Classes & Properties

A common workflow: load a public ontology, then add your own classes and properties to tailor it to your research domain.

Let's extend the GO slim by adding classes for specific biological pathways and a new property linking processes to their cellular locations.

In [32]:
# Add new subclasses under existing GO classes
# First, find existing classes we want to extend
immune_process = go.search_one(label="immune system process")
signaling = go.search_one(label="signaling")
cell_death = go.search_one(label="programmed cell death")

print(f"Extending: {immune_process.label[0]}")
print(f"Extending: {signaling.label[0]}")
print(f"Extending: {cell_death.label[0]}")

Extending: immune system process
Extending: signaling
Extending: programmed cell death


In [33]:
# Add new subclasses within the GO ontology's namespace
with go:
    # New immune process subclasses
    class T_cell_activation(immune_process):
        label = ["T cell activation"]
    
    class B_cell_activation(immune_process):
        label = ["B cell activation"]
    
    class cytokine_production(immune_process):
        label = ["cytokine production"]
    
    # New signaling subclasses
    class Wnt_signaling(signaling):
        label = ["Wnt signaling pathway"]
    
    class Notch_signaling(signaling):
        label = ["Notch signaling pathway"]
    
    # New cell death subclass
    class ferroptosis(cell_death):
        label = ["ferroptosis"]
        comment = ["Iron-dependent form of regulated cell death"]

print("New classes added to GO slim:")
for cls in [T_cell_activation, B_cell_activation, cytokine_production,
            Wnt_signaling, Notch_signaling, ferroptosis]:
    print(f"  {cls.label[0]} — subclass of {cls.is_a[0].label[0]}")

New classes added to GO slim:
  T cell activation — subclass of immune system process
  B cell activation — subclass of immune system process
  cytokine production — subclass of immune system process
  Wnt signaling pathway — subclass of signaling
  Notch signaling pathway — subclass of signaling
  ferroptosis — subclass of programmed cell death


In [34]:
# Add a new object property
with go:
    class occurs_in_tissue(ObjectProperty):
        label = ["occurs in tissue"]
        domain = [Thing]
        range = [Thing]
    
    # Create a simple tissue class hierarchy to use as range
    class Tissue(Thing):
        label = ["tissue"]
    
    class BoneMarrow(Tissue):
        label = ["bone marrow"]
    
    class Thymus(Tissue):
        label = ["thymus"]
    
    class LymphNode(Tissue):
        label = ["lymph node"]

print(f"New property: {occurs_in_tissue.label[0]}")
print(f"New classes: {[c.label[0] for c in Tissue.subclasses()]}")

New property: occurs in tissue
New classes: ['bone marrow', 'thymus', 'lymph node']


In [35]:
# Add restrictions to our new classes using the new property
with go:
    T_cell_activation.is_a.append(occurs_in_tissue.some(Thymus))
    B_cell_activation.is_a.append(occurs_in_tissue.some(BoneMarrow))

# Verify the restrictions
print("T cell activation:")
print(f"  is_a: {T_cell_activation.is_a}")
print()
print("B cell activation:")
print(f"  is_a: {B_cell_activation.is_a}")

T cell activation:
  is_a: [obo.GO_0002376, goslim_generic.occurs_in_tissue.some(goslim_generic.Thymus), goslim_generic.occurs_in_tissue.some(goslim_generic.Thymus)]

B cell activation:
  is_a: [obo.GO_0002376, goslim_generic.occurs_in_tissue.some(goslim_generic.BoneMarrow), goslim_generic.occurs_in_tissue.some(goslim_generic.BoneMarrow)]


**What we did:**
- Added subclasses under existing GO classes using standard Python class inheritance
- Created a new `ObjectProperty` with domain and range
- Added existential restrictions (`some`) linking our new classes to tissue types
- All additions live in the GO slim's namespace since we used `with go:`

The extended ontology now combines the original public content with our custom additions.

## Section 5: Loading a Second Ontology

Now let's load the Disease Ontology subset. Both ontologies coexist in owlready2's default world — each with its own namespace (IRI).

In [36]:
# Load the disease ontology subset
disease = get_ontology("file://data/disease_ontology_subset.owl").load()

print(f"Disease Ontology subset loaded")
print(f"  Base IRI: {disease.base_iri}")
print(f"  Classes: {len(list(disease.classes()))}")
print(f"  Annotation properties: {[p.name for p in disease.annotation_properties()]}")

Disease Ontology subset loaded
  Base IRI: http://purl.obolibrary.org/obo/doid/subsets/do_tutorial_slim.owl#
  Classes: 77
  Annotation properties: ['hasExactSynonym', 'hasDbXref', 'definition']


In [37]:
# Explore the disease hierarchy
cancer = disease.search_one(label="cancer")
print(f"Root: {cancer.label[0]}")
print(f"Direct subtypes of cancer:")
for child in cancer.subclasses():
    label = child.label[0] if child.label else child.name
    sub_count = len(list(child.subclasses()))
    print(f"  {label} ({sub_count} subtypes)")

Root: cancer
Direct subtypes of cancer:
  breast cancer (7 subtypes)
  hematologic cancer (5 subtypes)
  lung cancer (3 subtypes)
  brain cancer (22 subtypes)


In [38]:
# Look at a specific disease with its annotations
breast_cancer = disease.search_one(label="breast cancer")
print(f"Class: {breast_cancer.label[0]}")
print(f"  IRI: {breast_cancer.iri}")
print(f"  Parent: {breast_cancer.is_a}")

if breast_cancer.comment:
    print(f"  Definition: {breast_cancer.comment[0]}")

if hasattr(breast_cancer, 'hasExactSynonym') and breast_cancer.hasExactSynonym:
    print(f"  Synonyms: {breast_cancer.hasExactSynonym}")

if hasattr(breast_cancer, 'hasDbXref') and breast_cancer.hasDbXref:
    print(f"  Cross-references: {breast_cancer.hasDbXref}")

Class: breast cancer
  IRI: http://purl.obolibrary.org/obo/doid/subsets/do_tutorial_slim.owl#DOID_1612
  Parent: [disease_ontology_subset.DOID_162, disease_biology_integrated.has_disrupted_process.some(obo.GO_0012501), disease_biology_integrated.has_disrupted_process.some(goslim_generic.Wnt_signaling)]


In [39]:
# Explore deeper — breast cancer subtypes
print("Breast cancer subtypes:")
for sub in breast_cancer.subclasses():
    label = sub.label[0] if sub.label else sub.name
    print(f"  {label}")
    for subsub in sub.subclasses():
        sublabel = subsub.label[0] if subsub.label else subsub.name
        print(f"    {sublabel}")

Breast cancer subtypes:
  breast carcinoma
    breast lobular carcinoma
    breast ductal carcinoma
    breast adenocarcinoma
    nipple duct carcinoma
  estrogen-receptor positive breast cancer
  estrogen-receptor negative breast cancer
  Her2-receptor positive breast cancer
  Her2-receptor negative breast cancer
  triple-receptor negative breast cancer
  breast angiosarcoma


**Two ontologies, one world:**

At this point, owlready2's default world holds both ontologies. They have separate IRIs and namespaces, but classes from both are accessible. This is the starting point for merging.

## Section 6: Merging via Import

In OWL, `owl:imports` declares that one ontology depends on another. When ontology A imports ontology B, all of B's axioms become available in A.

Let's create a new "integrated" ontology that imports both the GO slim and the disease subset.

In [40]:
# Create a new ontology that will integrate both
integrated = get_ontology("http://example.org/ontology/disease_biology_integrated.owl")

# Declare imports — this makes all classes from both ontologies available
integrated.imported_ontologies.append(go)
integrated.imported_ontologies.append(disease)

print(f"Integrated ontology created")
print(f"  IRI: {integrated.base_iri}")
print(f"  Imports: {[o.base_iri for o in integrated.imported_ontologies]}")

Integrated ontology created
  IRI: http://example.org/ontology/disease_biology_integrated.owl#
  Imports: ['http://purl.obolibrary.org/obo/go/subsets/goslim_generic.owl#', 'http://purl.obolibrary.org/obo/doid/subsets/do_tutorial_slim.owl#', 'http://purl.obolibrary.org/obo/go/subsets/goslim_generic.owl#', 'http://purl.obolibrary.org/obo/doid/subsets/do_tutorial_slim.owl#']


**What `imported_ontologies` does:**
- Declares a formal OWL import relationship
- All classes, properties, and axioms from imported ontologies become visible
- The integrated ontology can reference classes from either source
- When saved, the import declarations are included in the OWL file

This is different from just loading two files — the import is a *semantic* relationship that tools like Protege will follow.

In [41]:
# Verify we can access classes from both ontologies through the integrated one
# Classes are accessible via their original ontology references
print("Accessing GO classes:")
print(f"  {immune_process.label[0]} — from GO")
print(f"  {T_cell_activation.label[0]} — our extension to GO")
print()
print("Accessing Disease classes:")
print(f"  {cancer.label[0]} — from Disease Ontology")
print(f"  {breast_cancer.label[0]} — from Disease Ontology")

Accessing GO classes:
  immune system process — from GO
  T cell activation — our extension to GO

Accessing Disease classes:
  cancer — from Disease Ontology
  breast cancer — from Disease Ontology


## Section 7: Cross-Ontology References

The real power of merging comes from creating *new relationships* between classes that originated in different ontologies. 

Our scenario: linking diseases to the biological processes that are disrupted in those diseases. This is a common pattern in translational bioinformatics.

In [42]:
# Define a new property in the integrated ontology that bridges the two domains
with integrated:
    class has_disrupted_process(ObjectProperty):
        label = ["has disrupted process"]
        comment = ["Links a disease to biological processes that are dysregulated"]
    
    class is_disrupted_in(ObjectProperty):
        label = ["is disrupted in"]
        inverse_property = has_disrupted_process

In [43]:
# Now create cross-references: diseases linked to GO biological processes
# These represent real biological knowledge

with integrated:
    # Breast cancer involves disrupted cell death and signaling
    breast_cancer.is_a.append(has_disrupted_process.some(cell_death))
    breast_cancer.is_a.append(has_disrupted_process.some(Wnt_signaling))
    
    # Leukemia involves disrupted immune processes and cell death
    leukemia = disease.search_one(label="leukemia")
    leukemia.is_a.append(has_disrupted_process.some(immune_process))
    leukemia.is_a.append(has_disrupted_process.some(cell_death))
    
    # Brain cancer involves disrupted signaling
    brain_cancer = disease.search_one(label="brain cancer")
    brain_cancer.is_a.append(has_disrupted_process.some(Notch_signaling))
    
    # Lung cancer involves disrupted cell death
    lung_cancer = disease.search_one(label="lung cancer")
    lung_cancer.is_a.append(has_disrupted_process.some(ferroptosis))

print("Cross-ontology restrictions added:")
print(f"  breast cancer has_disrupted_process: cell death, Wnt signaling")
print(f"  leukemia has_disrupted_process: immune system process, cell death")
print(f"  brain cancer has_disrupted_process: Notch signaling")
print(f"  lung cancer has_disrupted_process: ferroptosis")

Cross-ontology restrictions added:
  breast cancer has_disrupted_process: cell death, Wnt signaling
  leukemia has_disrupted_process: immune system process, cell death
  brain cancer has_disrupted_process: Notch signaling
  lung cancer has_disrupted_process: ferroptosis


In [44]:
# Verify the cross-references are in place
print("Breast cancer — full is_a list:")
for axiom in breast_cancer.is_a:
    print(f"  {axiom}")

print()
print("Leukemia — full is_a list:")
for axiom in leukemia.is_a:
    print(f"  {axiom}")

Breast cancer — full is_a list:
  disease_ontology_subset.DOID_162
  disease_biology_integrated.has_disrupted_process.some(obo.GO_0012501)
  disease_biology_integrated.has_disrupted_process.some(goslim_generic.Wnt_signaling)
  disease_biology_integrated.has_disrupted_process.some(obo.GO_0012501)
  disease_biology_integrated.has_disrupted_process.some(goslim_generic.Wnt_signaling)

Leukemia — full is_a list:
  disease_ontology_subset.DOID_2531
  disease_biology_integrated.has_disrupted_process.some(obo.GO_0002376)
  disease_biology_integrated.has_disrupted_process.some(obo.GO_0012501)
  disease_biology_integrated.has_disrupted_process.some(obo.GO_0002376)
  disease_biology_integrated.has_disrupted_process.some(obo.GO_0012501)


In [45]:
# Query: which diseases have disrupted processes related to cell death?
print("Diseases with disrupted cell death processes:")
print("-" * 50)
for cls in disease.classes():
    for restriction in cls.is_a:
        if hasattr(restriction, 'property') and restriction.property == has_disrupted_process:
            if hasattr(restriction, 'value'):
                # Check if the value is cell_death or a subclass of it
                if restriction.value == cell_death or (hasattr(restriction.value, 'is_a') and cell_death in restriction.value.is_a):
                    label = cls.label[0] if cls.label else cls.name
                    proc_label = restriction.value.label[0] if restriction.value.label else restriction.value.name
                    print(f"  {label} → {proc_label}")

Diseases with disrupted cell death processes:
--------------------------------------------------
  breast cancer → programmed cell death
  breast cancer → programmed cell death
  leukemia → programmed cell death
  leukemia → programmed cell death
  lung cancer → ferroptosis
  lung cancer → ferroptosis


**Cross-reference patterns:**
- Define bridging properties in the integrated ontology (not in either source)
- Use `some` (existential) restrictions for "this disease involves disruption of that process"
- The inverse property (`is_disrupted_in`) lets you query in both directions
- These are proper OWL axioms — a reasoner can use them for inference

## Section 8: Reasoning & Exporting the Merged Ontology

Finally, we can run a reasoner over the merged ontology and export the result for use in Protege or other tools.

In [46]:
# First, let's check consistency with the reasoner
# This requires Java to be installed
try:
    with integrated:
        sync_reasoner(infer_property_values=True)
    print("Reasoning complete — ontology is consistent!")
    print()
    
    # Check if any new inferences were made
    print("Inferred statements (if any):")
    for cls in disease.classes():
        if cls.equivalent_to:
            print(f"  {cls.label[0] if cls.label else cls.name} equivalent_to: {cls.equivalent_to}")
            
except Exception as e:
    print(f"Reasoner unavailable (Java may not be installed): {type(e).__name__}")
    print("The ontology can still be saved and opened in Protege for reasoning.")
    print()
    print("To install Java: brew install openjdk (macOS) or apt install default-jdk (Linux)")
    print("-" * 50)
    print("Error:", e)

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /Users/justinadjasu/micromamba/envs/BMDSOWL/lib/python3.8/site-packages/owlready2/hermit:/Users/justinadjasu/micromamba/envs/BMDSOWL/lib/python3.8/site-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////var/folders/pg/n9hclc2d3_g0k___j1q033gw0000gn/T/tmp8m54tcxg -Y


Reasoning complete — ontology is consistent!

Inferred statements (if any):


* Owlready2 * HermiT took 0.547929048538208 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


In [47]:
# Save the integrated ontology
os.makedirs("ontologies", exist_ok=True)

# Save in RDF/XML format (most compatible with Protege)
integrated.save(file="ontologies/disease_biology_integrated.owl", format="rdfxml")
print(f"Saved: ontologies/disease_biology_integrated.owl")
print(f"  Size: {os.path.getsize('ontologies/disease_biology_integrated.owl'):,} bytes")

# Also save the extended GO slim separately
go.save(file="ontologies/goslim_extended.owl", format="rdfxml")
print(f"Saved: ontologies/goslim_extended.owl")
print(f"  Size: {os.path.getsize('ontologies/goslim_extended.owl'):,} bytes")

Saved: ontologies/disease_biology_integrated.owl
  Size: 3,324 bytes
Saved: ontologies/goslim_extended.owl
  Size: 511,484 bytes


In [48]:
# Final summary of what we built
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print()
print("Source ontologies:")
print(f"  GO slim:          {len(list(go.classes()))} classes, {len(list(go.object_properties()))} properties")
print(f"  Disease subset:   {len(list(disease.classes()))} classes")
print()
print("Integrated ontology includes:")
print(f"  - All GO slim classes + our extensions (tissues, pathways)")
print(f"  - All Disease Ontology cancer classes")
print(f"  - Bridging property: has_disrupted_process / is_disrupted_in")
print(f"  - Cross-ontology restrictions linking diseases to processes")
print()
print("Output files (in ontologies/):")
print(f"  - disease_biology_integrated.owl  (merged ontology with imports)")
print(f"  - goslim_extended.owl             (GO slim + our additions)")
print()
print("Next steps:")
print("  - Open in Protege to visualize the merged hierarchy")
print("  - Add more cross-references based on literature")
print("  - Run a reasoner in Protege for full classification")

SUMMARY

Source ontologies:
  GO slim:          150 classes, 10 properties
  Disease subset:   77 classes

Integrated ontology includes:
  - All GO slim classes + our extensions (tissues, pathways)
  - All Disease Ontology cancer classes
  - Bridging property: has_disrupted_process / is_disrupted_in
  - Cross-ontology restrictions linking diseases to processes

Output files (in ontologies/):
  - disease_biology_integrated.owl  (merged ontology with imports)
  - goslim_extended.owl             (GO slim + our additions)

Next steps:
  - Open in Protege to visualize the merged hierarchy
  - Add more cross-references based on literature
  - Run a reasoner in Protege for full classification


## Summary

In this tutorial, you learned how to:

1. **Load** public ontologies from URLs and local files using `get_ontology().load()`
2. **Explore** an ontology's classes, properties, and annotations
3. **Extend** an existing ontology by adding subclasses, properties, and restrictions
4. **Import** multiple ontologies into a single integrated ontology
5. **Create cross-references** between classes from different ontologies using bridging properties
6. **Export** the merged result for use in other tools

These patterns form the basis of ontology integration work in biomedical informatics — combining resources like the Gene Ontology, Disease Ontology, Human Phenotype Ontology, and ChEBI to build richer knowledge representations for your specific research questions.